# Spin-basis 2D — ansatz comparisonSweep `2026-09-28_15-04-11G50d5e6fP3681539`: four wavefunctions on the same6x6 $J_1$ model (`J2 = 0`, `sign_rule = FF`), zipped over`model.type` x `model.rbm.alpha`:| run | ansatz | n_params ||---|---|---|| 0 | Jastrow | 630 || 1 | RBM, $\alpha$ = 1 | 1 368 || 2 | RBM, $\alpha$ = 2 | 2 700 || 3 | ViT (2 layers, d=72, 12 heads, patch 2) | 116 784 |Sampling is plain Born (`auto_alpha = false`, $\alpha$ = 2), so there is noimportance-sampling reweighting here and `ESS_w` sits at 100% by construction —the diagnostics worth watching are the local-energy variance, R-hat and theMetropolis acceptance.Data layout: one `run_000X/` per ansatz. Logs are NetKet `JsonLog` files named`run step NNNNNN.log`, one per resubmission chunk, with *global* iterationnumbers — so every `*.log` in a run dir is merged and sorted. Sampling health isappended to a single `run.diag.jsonl`.

In [ ]:
from pathlib import Pathimport json, reimport numpy as npimport matplotlib.pyplot as pltdef _data_root(name="spinbasis_data"):    here = Path.cwd().resolve()    for base in (here, *here.parents):        if (base / name).is_dir():            return base / name    raise FileNotFoundError(f"no {name!r} directory at or above {here}")DATA_ROOT = _data_root()# Newest sweep folder by name (the timestamp prefix sorts chronologically).# Set SWEEP_DIR by hand to pin an older one.SWEEP_DIR = sorted(p for p in DATA_ROOT.iterdir()                   if p.is_dir() and (p / "matrix.json").exists())[-1]# 6x6, J2 = 0 reference (per site). ED is skipped at this size, so E0 is null in# run.summary.json and this literal is the only reference available.E0_PER_SITE = -0.678872SMOOTH = 101        # rolling-mean window, in steps, for the trend linesprint("sweep:", SWEEP_DIR.name)

## Load

In [ ]:
def _merge_logs(run_dir):    """Every *.log in a run dir -> one curve sorted by global iteration."""    series = {}    for lp in sorted(run_dir.glob("*.log")):        E = json.load(open(lp))["Energy"]        for i, m, s, v, rh in zip(E["iters"], E["Mean"]["real"], E["Sigma"],                                  E["Variance"], E["R_hat"]):            series[int(i)] = (m, s, v, rh)    it = np.array(sorted(series))    cols = np.array([series[i] for i in it], dtype=float)    return it, cols[:, 0], cols[:, 1], cols[:, 2], cols[:, 3]def _diag(run_dir):    """run.diag.jsonl -> dict of arrays (append-only, global step)."""    p = run_dir / "run.diag.jsonl"    if not p.exists():        return {}    rows = [json.loads(l) for l in p.open() if l.strip()]    keys = ("step", "var_eloc_per_site", "ess_w_frac", "rhat_logpsi", "acceptance")    return {k: np.array([r.get(k, np.nan) for r in rows], dtype=float) for k in keys}def _from_stdout(run_dir):    """The driver prints the ansatz and its parameter count; summary.json only    exists for finished runs, so take both from stdout instead."""    txt = (run_dir / "stdout.txt").read_text(errors="replace")    wf = re.search(r"^wavefunction:\s*(.+)$", txt, re.M)    npar = re.search(r"n_params=(\d+)", txt)    return (wf.group(1).strip() if wf else "?",            int(npar.group(1)) if npar else None)def _status(run_dir):    if (run_dir / "run.done").exists():        return "completed"    if (run_dir / "run.timeout").exists():        return "timeout"    return "in progress"        # neither flag written yetmatrix = json.load(open(SWEEP_DIR / "matrix.json"))runs = []for spec in matrix["runs"]:    run_dir = SWEEP_DIR / f"run_{spec['run_id']:04d}"    it, E, sigma, var, rhat = _merge_logs(run_dir)    cfg = spec["config"]    wf, n_params = _from_stdout(run_dir)    mtype = spec["sweep_values"]["model.type"]    rbm_alpha = spec["sweep_values"]["model.rbm.alpha"]    runs.append({        "id": spec["run_id"], "dir": run_dir, "type": mtype,        "rbm_alpha": rbm_alpha, "wf": wf, "n_params": n_params,        "n_sites": cfg["system"]["L"] ** 2, "status": _status(run_dir),        "target": cfg["model"]["iter"],        "iters": it, "E": E, "sigma": sigma, "var": var, "rhat": rhat,        "diag": _diag(run_dir),    })# Colour carries the ansatz family; dash separates the two RBM widths. Three# hues instead of four keeps the families distinguishable at a glance.COLOR = {"jastrow": "#2a78d6", "rbm": "#eb6834", "vit": "#1baf7a"}SOLID, DASHED = (0, ()), (0, (5, 2))def tag(r):    """Short direct-label tag (the legend carries the full name)."""    if r["type"] == "rbm":        return rf"RBM $\alpha$={r['rbm_alpha']:g}"    return "ViT" if r["type"] == "vit" else "Jastrow"def label(r):    if r["type"] == "rbm":        return rf"RBM $\alpha$={r['rbm_alpha']:g}  ({r['n_params']:,}p)"    return f"{r['type'].capitalize() if r['type'] != 'vit' else 'ViT'}  ({r['n_params']:,}p)"for r in runs:    r["color"] = COLOR[r["type"]]    r["dash"] = DASHED if (r["type"] == "rbm" and r["rbm_alpha"] == 2.0) else SOLID    print(f"run_{r['id']:04d}  {r['wf'][:60]:60s}  {len(r['iters']):5d}"          f"/{r['target']} steps  {r['status']}")

## Final numbers`E/site` is the mean over the last 200 logged steps — a single VMC step is anoisy estimator, so the last point alone is not a run's energy.

In [ ]:
TAIL = 200print(f"{'run':>4} {'ansatz':>16} {'n_params':>9} {'steps':>6} {'status':>12} "      f"{'E/site':>11} {'+-std':>9} {'rel.err':>9} {'Var/site':>10} {'acc':>6}")for r in runs:    e_tail = r["E"][-TAIL:] / r["n_sites"]    e, sd = e_tail.mean(), e_tail.std()    rel = abs(e - E0_PER_SITE) / abs(E0_PER_SITE)    d = r["diag"]    acc = np.nanmean(d["acceptance"][-10:]) if d else np.nan    name = (rf"RBM a={r['rbm_alpha']:g}" if r["type"] == "rbm"            else ("ViT" if r["type"] == "vit" else "Jastrow"))    print(f"{r['id']:>4} {name:>16} {r['n_params']:>9,} {len(r['iters']):>6} "          f"{r['status']:>12} {e:>11.6f} {sd:>9.1e} {rel:>9.2e} "          f"{r['var'][-TAIL:].mean() / r['n_sites']:>10.2e} {acc:>6.2f}")print(f"\nreference E0/site = {E0_PER_SITE:.6f}  (6x6, J2 = 0)")

## Energy per site vs. iterationRaw trace at low opacity, centred rolling mean on top. The left panel uses a logiteration axis — on a linear axis the whole descent collapses into the firstpixel column. The right panel zooms onto the plateaus of the three runs thatconverge near the reference.

In [ ]:
def smooth(y, w):    w = min(w, max(1, len(y) // 4) * 2 + 1)    if w < 3:        return y    k = np.ones(w) / w    pad = w // 2    return np.convolve(np.pad(y, pad, mode="edge"), k, mode="valid")fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6),                         gridspec_kw={"width_ratios": [1.25, 1]})for ax in axes:    for r in runs:        y = r["E"] / r["n_sites"]        ax.plot(r["iters"], y, color=r["color"], lw=0.6, alpha=0.18)        ax.plot(r["iters"], smooth(y, SMOOTH), color=r["color"], lw=1.8,                dashes=r["dash"][1], label=label(r), solid_capstyle="round")    ax.axhline(E0_PER_SITE, color="0.35", ls=":", lw=1.2, zorder=0)    ax.set_xlabel("iteration")    ax.grid(alpha=0.2, lw=0.6)    ax.set_axisbelow(True)    for sp in ("top", "right"):        ax.spines[sp].set_visible(False)axes[0].set_xscale("log")axes[0].set_xlim(1, 1.2e4)axes[0].set_ylim(-0.70, -0.25)axes[0].set_ylabel(r"energy per site  $\langle H\rangle / N$")axes[0].set_title("full history (log iteration)", fontsize=10)axes[0].legend(fontsize=8.5, loc="upper right", frameon=False)axes[0].annotate(f"$E_0/N$ = {E0_PER_SITE:.6f}", xy=(0.02, E0_PER_SITE),                 xycoords=("axes fraction", "data"), ha="left", va="bottom",                 fontsize=8, color="0.35")axes[1].set_xlim(0, 1.03e4)axes[1].set_ylim(-0.6795, -0.6700)axes[1].xaxis.set_major_locator(plt.MaxNLocator(5))axes[1].set_title("zoom on the converged plateaus", fontsize=10)axes[1].annotate(f"$E_0/N$ = {E0_PER_SITE:.6f}", xy=(0.99, E0_PER_SITE),                 xycoords=("axes fraction", "data"), ha="right", va="bottom",                 fontsize=8, color="0.35")# direct-label the plateaus, staggered so near-coincident ends stay readableends = sorted(((r["iters"][-1], smooth(r["E"] / r["n_sites"], SMOOTH)[-1],                r["color"], tag(r)) for r in runs), key=lambda t: t[1])for rank, (x, y, c, txt) in enumerate(ends):    if -0.6795 < y < -0.6700:        axes[1].annotate(txt, xy=(x, y), xytext=(5, -6 + 6 * (rank % 2)),                         textcoords="offset points", ha="left", va="center",                         fontsize=8.5, color=c)fig.suptitle("VMC energy per site — 6x6 $J_1$, spin basis", x=0.01, ha="left",             fontsize=12, y=1.02)fig.tight_layout()plt.show()

## Relative error vs. iteration$|E - E_0| / |E_0|$ against the per-site reference — scale-invariant, sototal-vs-per-site is irrelevant. Smoothed only: the raw quantity spikes on a logaxis wherever the trace crosses the reference.

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.8))ends = []for r in runs:    rel = smooth(np.abs(r["E"] / r["n_sites"] - E0_PER_SITE) / abs(E0_PER_SITE),                 SMOOTH)    ax.loglog(r["iters"], rel, color=r["color"], lw=1.8, dashes=r["dash"][1],              label=label(r), solid_capstyle="round")    ends.append((r["iters"][-1], rel[-1], r["color"]))for rank, (x, y, c) in enumerate(sorted(ends, key=lambda t: t[1])):    ax.annotate(f"{y:.1e}", xy=(x, y), xytext=(6, -9 + 9 * (rank % 2)),                textcoords="offset points", ha="left", va="center",                fontsize=8.5, color=c)ax.set(xlabel="iteration", ylabel=r"$|E - E_0| \,/\, |E_0|$",       title="Relative error vs. iteration")ax.margins(x=0.12)ax.grid(alpha=0.2, lw=0.6, which="both")ax.set_axisbelow(True)for sp in ("top", "right"):    ax.spines[sp].set_visible(False)ax.legend(fontsize=8.5, loc="lower left", frameon=False)fig.tight_layout()plt.show()

## Sampling healthFrom `run.diag.jsonl`, one panel per quantity (shared x, never a second y-axis).`Var(E_loc)/N` is the variational-quality indicator — it goes to zero only for anexact eigenstate. A run whose acceptance falls to zero *and* whose variancecollapses to machine precision has frozen on a single configuration rather thanconverged; watch the two panels together.

In [ ]:
# ESS_w is omitted on purpose: with auto_alpha = false the weights are all 1 and# the panel is a flat line at 100% for every run.PANELS = [    # The frozen Jastrow drives Var/site to ~1e-31; clipping keeps the band the    # other three actually occupy readable, and the run is flagged below.    ("var_eloc_per_site",  r"Var$(E_{loc})/N$",  "log",  (1e-6, 3e0)),    ("acceptance",         "acceptance",         None,   (0, 0.6)),    ("rhat_logpsi",        r"$\hat{R}$  $\log|\psi|$",  None,  None),]DIAG_SMOOTH = 9     # diag rows land every 20 steps -> ~180 stepsfig, axes = plt.subplots(len(PANELS), 1, figsize=(9, 7.4), sharex=True)for ax, (key, ylab, scale, ylim) in zip(axes, PANELS):    for r in runs:        d = r["diag"]        if not d:            continue        y = d[key]        ax.plot(d["step"], y, color=r["color"], lw=0.7, alpha=0.18)        ax.plot(d["step"], smooth(y, DIAG_SMOOTH), color=r["color"], lw=1.6,                dashes=r["dash"][1], label=label(r), solid_capstyle="round")    if scale:        ax.set_yscale(scale)    if ylim:        ax.set_ylim(*ylim)    ax.set_ylabel(ylab)    ax.grid(alpha=0.2, lw=0.6)    ax.set_axisbelow(True)    for sp in ("top", "right"):        ax.spines[sp].set_visible(False)axes[2].axhline(1.0, color="0.35", ls=":", lw=1.2, zorder=0)axes[0].legend(fontsize=8.5, loc="upper right", ncol=2, frameon=False)# call out any run that froze: acceptance at zero with a machine-precision# variance is a collapsed wavefunction, not a converged onefor r in runs:    d = r["diag"]    if d and np.nanmean(d["acceptance"][-10:]) < 1e-3:        step = d["step"][np.argmax(d["var_eloc_per_site"] < 1e-6)]        axes[0].annotate(f"{tag(r)} froze at step ~{step:.0f}: acc -> 0,\n"                         f"Var/site ~ {d['var_eloc_per_site'][-1]:.0e} (off-scale)",                         xy=(step, 1.6e-6), xycoords="data",                         xytext=(0.17, 0.04), textcoords="axes fraction",                         fontsize=8, color=r["color"], ha="left", va="bottom",                         arrowprops=dict(arrowstyle="-", lw=0.8,                                         color=r["color"], alpha=0.7))axes[-1].set_xlabel("iteration")fig.suptitle("Sampling diagnostics", x=0.01, ha="left", fontsize=12, y=0.995)fig.tight_layout()plt.show()